In [ ]:
"""
================================================================================
Project: Global Mangrove Watch - Annual Sea Level Rise Extraction (ECMWF ORAS5)
Description:
    Extracts monthly Mean Sea Level (MSL) data from ECMWF ocean reanalysis
    NetCDF files (.nc), filters out land-mask contamination, matches each
    mangrove tile centroid to its nearest valid ocean grid cell, computes
    annual averages, and exports the results in a tidy long format separated
    by semicolons (;).

Inputs:
    - GeoJSON file containing mangrove tile polygons and a 'tilename' field.
    - Directory tree containing ECMWF NetCDF ocean reanalysis files.

Outputs:
    - Annual mean MSL CSV file (`ecmwf_msl_annual_1deg_tiles.csv`) in long format.
================================================================================
"""

import os
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import xarray as xr

# ==============================================================================
# 1. DIRECTORY & PATH CONFIGURATION
# ==============================================================================
# Override these paths via environment variables or update them for your local machine
DIR_MANGROVES = Path(os.getenv("GMW_DIR", r"D:\MANGROVES\GMW\V4"))
GEOJSON_PATH = Path(os.getenv("GMW_TILES_GEOJSON", DIR_MANGROVES / "gmw_v4112_tiles.geojson"))

DIR_MSL = Path(os.getenv("GMW_SSH_DIR", r"D:\MANGROVES\SSH"))

# Directory sources for ECMWF NetCDF model outputs
FOLDERS = [
    DIR_MSL / "ECMWF_1",
    DIR_MSL / "ECMWF_2",
    DIR_MSL / "ECMWF_3",
]

# ==============================================================================
# 2. LOAD TILE POLYGONS FROM GEOJSON & RENAME tilename -> tile_id
# ==============================================================================
print("Loading mangrove tile polygons from GeoJSON...")
gdf_tiles = gpd.read_file(GEOJSON_PATH)

# Check if 'tilename' exists and rename it to 'tile_id'
if "tilename" in gdf_tiles.columns:
    print("Renaming column 'tilename' to 'tile_id'...")
    gdf_tiles = gdf_tiles.rename(columns={"tilename": "tile_id"})
elif "tile_id" not in gdf_tiles.columns:
    raise KeyError("Error: Neither 'tilename' nor 'tile_id' found in the GeoJSON columns!")

# Ensure the GeoDataFrame is in WGS84 (EPSG:4326) for geographic coordinate calculations
if gdf_tiles.crs is not None and gdf_tiles.crs.to_string() != "EPSG:4326":
    print("Reprojecting tiles to EPSG:4326 (WGS84)...")
    gdf_tiles = gdf_tiles.to_crs(epsg=4326)

# Compute polygon centroids to derive center coordinates for each tile
print("Calculating geographic centroids (center_lon, center_lat) for each tile...")
gdf_tiles["centroid"] = gdf_tiles.geometry.centroid
gdf_tiles["center_lon"] = gdf_tiles["centroid"].x
gdf_tiles["center_lat"] = gdf_tiles["centroid"].y

# Extract unique tiles and their computed central coordinates
coords_df = (
    gdf_tiles[["tile_id", "center_lon", "center_lat"]]
    .drop_duplicates(subset=["tile_id"])
    .sort_values("tile_id")
)

tile_ids = coords_df["tile_id"].values
lons_wgs84 = coords_df["center_lon"].values
lats_wgs84 = coords_df["center_lat"].values

print(f"Successfully loaded and processed {len(tile_ids)} unique mangrove tiles.")

# ==============================================================================
# 3. EXTRACTION ON OCEAN GRID (MASKING LAND & NEAREST-NEIGHBOR MATCHING)
# ==============================================================================
monthly_records = []

for folder in FOLDERS:
    if not folder.exists():
        print(f"Warning: Directory not found -> {folder}. Skipping.")
        continue

    nc_files = sorted(list(folder.glob("**/*.nc")))
    print(f"\nProcessing directory: {folder.name} ({len(nc_files)} NetCDF files found)...")

    for nc_path in nc_files:
        print(f"  -> Reading: {nc_path.name}")

        with xr.open_dataset(nc_path, engine="netcdf4") as ds:
            # Detect variable name dynamically ('sossheig' for NEMO/ORAS5, 'zos' or fallback)
            var_name = (
                "sossheig"
                if "sossheig" in ds
                else ("zos" if "zos" in ds else list(ds.data_vars)[0])
            )

            # Detect time dimension name across different ECMWF conventions
            time_dim = "time" if "time" in ds.coords else "time_counter"
            time_val = ds[time_dim].values[0]

            # Extract 2D grid coordinates
            lon_var = "nav_lon" if "nav_lon" in ds else "longitude"
            lat_var = "nav_lat" if "nav_lat" in ds else "latitude"

            ds_lon = ds[lon_var].values
            ds_lat = ds[lat_var].values

            # Handle longitude convention mapping ([0, 360] vs [-180, 180])
            if ds_lon.max() > 180:
                lons_query = np.where(lons_wgs84 < 0, lons_wgs84 + 360, lons_wgs84)
            else:
                lons_query = lons_wgs84

            # Extract 2D ocean variable matrix for the current time step
            data_2d = ds[var_name].squeeze().values

            # Create a boolean mask to exclude land cells (NaN values)
            valid_mask = ~np.isnan(data_2d)

            if not np.any(valid_mask):
                print(f"    Warning: All grid cells are NaN in {nc_path.name}. Filling with NaNs.")
                vals = [np.nan] * len(tile_ids)
            else:
                # Isolate coordinates and values of valid ocean-only cells
                ocean_lons = ds_lon[valid_mask]
                ocean_lats = ds_lat[valid_mask]
                ocean_vals = data_2d[valid_mask]

                # Find the closest valid ocean grid cell for each tile centroid using Euclidean distance
                vals = []
                for target_lon, target_lat in zip(lons_query, lats_wgs84):
                    dist = (ocean_lons - target_lon) ** 2 + (ocean_lats - target_lat) ** 2
                    nearest_idx = np.argmin(dist)
                    vals.append(float(ocean_vals[nearest_idx]))

            # Store the extracted record for this timestamp
            row = {"time": time_val}
            for tid, val in zip(tile_ids, vals):
                row[tid] = val

            monthly_records.append(row)

# ==============================================================================
# 4. ANNUAL AGGREGATION & TIDY FORMAT EXPORT (; SEPARATED)
# ==============================================================================
print("\nAssembling temporal records, computing annual means, and restructuring to long format...")
df_monthly = pd.DataFrame(monthly_records)
df_monthly["time"] = pd.to_datetime(df_monthly["time"])

# Clean up duplicates and sort chronologically
df_monthly = df_monthly.drop_duplicates(subset=["time"]).sort_values("time")

# Compute annual means grouped by year
df_monthly["year"] = df_monthly["time"].dt.year
df_annual = df_monthly.groupby("year")[tile_ids].mean().reset_index()

# Melt DataFrame from wide format to long format (tidy structure)
df_long = df_annual.melt(
    id_vars=["year"],
    value_vars=tile_ids,
    var_name="tile_id",
    value_name="SSH_m",
)

# Map center coordinates back to each tile_id row
coords_mapping = coords_df.set_index("tile_id")[["center_lat", "center_lon"]].to_dict("index")
df_long["center_lat"] = df_long["tile_id"].map(lambda x: coords_mapping[x]["center_lat"])
df_long["center_lon"] = df_long["tile_id"].map(lambda x: coords_mapping[x]["center_lon"])

# Reorder columns to match exact requested format: tile_id;year;center_lat;center_lon;SSH_m
df_long = df_long[["tile_id", "year", "center_lat", "center_lon", "SSH_m"]]

# Sort logically by tile and year
df_long = df_long.sort_values(["tile_id", "year"]).reset_index(drop=True)

# Export as semicolon-separated CSV
output_csv_annual = DIR_MSL / "ecmwf_msl_annual_1deg_tiles.csv"
df_long.to_csv(output_csv_annual, sep=";", index=False, float_format="%.4f")

# ==============================================================================
# 5. EXECUTION SUMMARY
# ==============================================================================
print("\nAnnual data extraction and tidy formatting completed successfully!")
print(f"  Output file (long format) : {output_csv_annual}")
print(f"  Total records exported    : {len(df_long)}")
print(f"  Temporal span             : {df_long['year'].min()} -> {df_long['year'].max()}")